# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields using their `@id` values.

**Note:** In Croissant, `recordSet` entities define the main tables (record sets) in the dataset, and each has a unique `@id` field. Fields/columns in each record set also have `@id`.


In [ ]:
# Inspect all available record sets and their fields by @id

record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record sets. Listing all with their fields (columns):\n")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields: [@id -- name -- description]")
    for field in rs.fields:
        print(f"    - {field.id} -- {field.name} -- {field.description}")
    print("\n")

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. Use the record set and field `@id`s obtained above.


In [ ]:
# Get all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record_set {record_set_id}")

if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"First record set id: {main_rs_id}")
    print(f"Columns: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    print("No tabular record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping data using field `@id`s.

Below, we demonstrate with the first numeric field found in the first (main) record set.

In [ ]:
# Identify a numeric field in the main record set using @id
import numpy as np

main_rs = None
for rs in metadata.record_sets:
    if rs.id == main_rs_id:
        main_rs = rs
        break

numeric_field_id = None
group_field_id = None
for field in main_rs.fields:
    # Try to find a float or integer field
    dt = field.data_type
    if numeric_field_id is None and (dt and ('Float' in dt or 'Integer' in dt or "Number" in dt)):
        if field.id in dataframes[main_rs_id].columns:
            numeric_field_id = field.id
    # Try to find a possible group field (categorical)
    if group_field_id is None and (dt and ("Text" in dt or "String" in dt)):
        if field.id in dataframes[main_rs_id].columns:
            group_field_id = field.id
    if numeric_field_id and group_field_id:
        break

if numeric_field_id:
    print(f"Numeric field (for filtering/normalizing): {numeric_field_id}")
    # Coerce column to numeric if needed
    df = dataframes[main_rs_id].copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Remove nan values
    df = df.dropna(subset=[numeric_field_id])

    threshold = np.percentile(df[numeric_field_id], 90)  # Use top 10% as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (top 10%):")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a likely categorical field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found in first record set; please inspect dataframes for analysis.")

## 5. Visualization
Visualize data distributions or relationships in the dataset. Example: histogram of the main numeric field and box plot by a group/categorical field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.show()

    # Box plot by group_field
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f'Boxplot of {numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion
In this notebook, we loaded a FAIR Croissant metadata-defined dataset on mass spectrometry analysis of human mast cell extracellular vesicle proteins, explored the record set structure via `@id`, extracted and explored data using field `@id`s, and performed basic EDA and visualization using `mlcroissant` and pandas. You may extend these analyses further for advanced proteomics or bioinformatics insights.
